In [4]:
import numpy as np

In [5]:
def shift(arr, lag):
    shifted = np.roll(arr, lag, axis=1)
    if lag >0:
        shifted[:, 0:lag] = 0
    else:
        shifted[:, lag:] = 0
    return shifted

In [6]:
arr = np.array([
                [0,0,0,0,0, 1,1,1,1,1, 0,0,0,0,0, 0,0,0,0,0, 1,1,1,1,1, 1,1,1,1,1, 0,0,0,0,0],
                [0,0,0,0,0, 1,1,1,1,1, 0,0,0,0,0, 0,0,0,0,0, 1,1,1,1,1, 1,1,1,1,1, 0,0,0,0,0]
                ])


In [7]:
cut = 2.9
step = 1

def _erosion(arr, cut, step):
    """
    Function to erode mask array for vignetting.
        2D matrix erosion for simulating finite thickness effect in shadow projections.
    It takes a mask array and "thins" the mask elements across the columns' direction.
    The erosion is performed only on the correct side of open (1) mask elements:
        right side if cut is negative (= thetaX negative)
        left side if cut is positive (= thetaX positive)
    The function first erodes all integer bins (replacing 1s with 0s)
    If cut is not integer, then the function applies a fractional transparency to the last eroded bin

              \       \  \
    ___________\       \  \____________
               |\       \ |
    ___________| \       \|_____________
                  \       \  \ 
                   \       \  \
    ________________\_______\__\_________
    <--------------->        <->
          SHIFT             EROSION   
    
    """
    ncuts = int(cut / step) #find the integer numbers of bins to cut
    decimal = abs(cut/step - ncuts) #find the decimal number of bins to cut

    #Find arr indexes to be cut (completely)
    shifted = shift(arr, ncuts)

    eroded_int = arr * ( (arr-shifted) > 0)

    print("\noriginal")
    print(*arr[0, :])
    print(*arr[-1, :])

    print("\nerode idx")
    print(*eroded_int[0, :])
    print(*eroded_int[-1, :]) 

    #Finds arr indexes to be fractionally reduced
    if decimal:
        shifted_p1 = shift(arr, ncuts + np.sign(ncuts))
        eroded_frac = arr * ((shifted_p1 | ~shifted)<-1)
    else:
        eroded_frac = arr * 0

    print("\nfract idx")
    print(*eroded_frac[0, :])
    print(*eroded_frac[-1, :])

    #Calculate output array
    out = (arr * (eroded_int < 1)) - eroded_frac * decimal

    print()
    print("\nresult")
    print(*out[0, :])
    print(*out[-1, :])

    return out


<>:5: SyntaxWarning: invalid escape sequence '\ '
<>:5: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_29307/2076972229.py:5: SyntaxWarning: invalid escape sequence '\ '
  """


In [8]:
cut = 2.9
step = 1

_erosion(arr, cut, step)


original
0 0 0 0 0 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0
0 0 0 0 0 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0

erode idx
0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0

fract idx
0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0


result
0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.10000000000000009 1.0 1.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.10000000000000009 1.0 1.0 1.0 1.0 1.0 1.0 1.0 0.0 0.0 0.0 0.0 0.0
0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.10000000000000009 1.0 1.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.10000000000000009 1.0 1.0 1.0 1.0 1.0 1.0 1.0 0.0 0.0 0.0 0.0 0.0


array([[0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.1, 1. , 1. , 0. , 0. , 0. ,
        0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.1, 1. , 1. , 1. ,
        1. , 1. , 1. , 1. , 0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.1, 1. , 1. , 0. , 0. , 0. ,
        0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.1, 1. , 1. , 1. ,
        1. , 1. , 1. , 1. , 0. , 0. , 0. , 0. , 0. ]])

In [12]:
arr = np.array(
    [
        [1, 1, 1, 0, 0, 0, 1],
        [1, 1, 1, 0, 0, 0, 1],
        [1, 1, 1, 0, 0, 0, 1],
    ]
)

step = 0.5
cut = -0.25

_erosion(arr, cut, step)


original
1 1 1 0 0 0 1
1 1 1 0 0 0 1

erode idx
1 1 1 0 0 0 1
1 1 1 0 0 0 1

fract idx
0 0 0 0 0 0 0
0 0 0 0 0 0 0


result
0.0 0.0 0.0 0.0 0.0 0.0 0.0
0.0 0.0 0.0 0.0 0.0 0.0 0.0


array([[0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.]])